# 07.07 - Custom Dataset and DataLoader

**Daily output:** `ImageDataset` from a CSV with `image_path,label`, plus printed batch shapes and labels.

Today turns image files into model-ready batches: CSV metadata, label mapping, custom `Dataset`, transforms, `DataLoader`, train/validation behavior, imbalance checks, and visual sanity checks.

**Notebook type:** Practice notebook with theory, exercises, and TODO cells.


## Dataset/DataLoader Mental Model

A `Dataset` answers two questions: how many examples exist, and how do I load one example? A `DataLoader` handles batching, shuffling, workers, and stacking examples into tensors.

For image classification, a CSV usually has `image_path` and `label`. The model needs integer class IDs, so create a stable mapping like `{'cat': 0, 'dog': 1}`.


In [ ]:
from pathlib import Path
import csv
import random

import numpy as np
from PIL import Image, ImageDraw
import torch
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed. Visualization examples will be skipped.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path("_day07_image_data")
IMG_DIR = ROOT / "images"
CSV_PATH = ROOT / "labels.csv"
IMG_DIR.mkdir(parents=True, exist_ok=True)


## Create a Tiny Image Dataset

This notebook generates a synthetic image dataset so it runs without downloads. In a contest, replace this with the provided CSV and image folder.


In [ ]:
def make_colored_image(label, size=(80, 80), seed=0):
    rng = np.random.default_rng(seed)
    base = {
        "red": np.array([220, 40, 40], dtype=np.uint8),
        "green": np.array([40, 180, 80], dtype=np.uint8),
        "blue": np.array([50, 90, 220], dtype=np.uint8),
    }
    arr = np.zeros((size[1], size[0], 3), dtype=np.uint8)
    arr[...] = base[label]
    noise = rng.normal(0, 18, size=arr.shape).astype(np.int16)
    arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr, mode="RGB")
    ImageDraw.Draw(img).rectangle([10, 10, 30, 30], outline=(255, 255, 255), width=2)
    return img

counts = {"red": 16, "green": 10, "blue": 6}
rows = []
for label, count in counts.items():
    for i in range(count):
        path = IMG_DIR / f"{label}_{i:02d}.png"
        make_colored_image(label, seed=1000 + i).save(path)
        rows.append({"image_path": str(path), "label": label})
random.shuffle(rows)

with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["image_path", "label"])
    writer.writeheader()
    writer.writerows(rows)

print("wrote", CSV_PATH)
print("rows:", len(rows))
print(rows[:3])


## Read CSV and Build Label Mapping

The exact integer IDs do not matter as much as consistency. Save `label_to_id` and `id_to_label` with your experiment notes or checkpoint.


In [ ]:
# TODO 07-A: Read CSV and build label mapping.
# Implement read_label_csv(csv_path).
# Build:
# - samples
# - unique_labels
# - label_to_id
# - id_to_label

def read_label_csv(csv_path):
    raise NotImplementedError

# TODO: read CSV_PATH and print first row plus mappings.


## Minimal Transforms

Writing these once makes the image contract memorable: PIL RGB image in, `[C, H, W]` float tensor out.


In [ ]:
# TODO 07-B: Minimal transforms.
# Implement Resize, ToTensor, Normalize, Compose.
# Contract: PIL RGB image -> CHW float tensor.

class Resize:
    pass

class ToTensor:
    pass

class Normalize:
    pass

class Compose:
    pass

# TODO: create train_transform and val_transform.


## Build `ImageDataset`

Keep the Dataset focused: load one row, open image, transform it, map the label, and return useful metadata for debugging.


In [ ]:
# TODO 07-C: Build ImageDataset.
# Implement __len__ and __getitem__.
# Return a dict with image, label, label_name, path.

class ImageDataset(Dataset):
    def __init__(self, rows, label_to_id, transform=None, path_col="image_path", label_col="label"):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def __getitem__(self, idx):
        raise NotImplementedError

# TODO: instantiate full_ds and inspect one item.


## DataLoader Batches

The default collate function stacks tensors and keeps strings as lists/tuples. For images, your expected batch shape is `[B, C, H, W]`.


In [ ]:
# TODO 07-D: DataLoader batches.
# Create DataLoader(full_ds, batch_size=8, shuffle=True, num_workers=0).
# Print batch keys, image shape, label shape/dtype, label names, and paths.

raise NotImplementedError("Create and inspect a DataLoader batch.")


## Visual Batch Sanity Check

Always visualize a batch before training. It catches wrong labels, wrong colors, broken normalization, and accidental path mistakes.


In [ ]:
# TODO 07-E: Visual batch sanity check.
# Implement unnormalize_for_display(tensor).
# If matplotlib is available, plot up to 8 images with labels.

def unnormalize_for_display(tensor, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
    raise NotImplementedError

# TODO: visualize one batch.


## Train/Validation Split

For a tiny demo, `random_split` is fine. For real competitions, use a stratified split so every class appears in both train and validation.


In [ ]:
# TODO 07-F: Train/validation split.
# Use random_split with a fixed generator.
# Create train_loader with shuffle=True and val_loader with shuffle=False.
# Print sizes and one batch shape from each.

raise NotImplementedError("Create train/validation loaders.")


## Imbalance Check

Count classes before training. If classes are imbalanced, consider class-weighted loss, a sampler, more data, or metric-aware thresholding.


In [ ]:
# TODO 07-G: Imbalance check.
# Count labels.
# Build optional WeightedRandomSampler.
# Print one sampled batch of label names.

def count_labels(rows):
    raise NotImplementedError

# TODO: create sampler and balanced_loader.


## Robust Dataset Variant

During development, failing fast is best. For long training runs, you may choose a placeholder image and keep an error flag.


In [ ]:
# TODO 07-H: Robust Dataset variant.
# Extend ImageDataset.
# If strict=True, raise image loading errors.
# If strict=False, return a black placeholder image and include load_error.

class RobustImageDataset(ImageDataset):
    pass

# TODO: test with a missing image path.


## Day 07 Checklist

Check CSV columns, image paths, label mapping, one dataset item, one DataLoader batch, train loader shuffle, validation loader no shuffle, transform differences, class counts, and a visual batch.
